In [1]:
import os
import sys
import json
import time
from uuid import UUID
from langchain_aws import ChatBedrock
from lunary import LunaryCallbackHandler
from langchain.schema import LLMResult
from langchain.prompts import PromptTemplate

from typing import Any, Union
from functools import wraps

LLMONITOR_API_URL = "http://54.172.197.178:3333"
LLMONITOR_APP_ID = "6e6c3fb6-2970-4231-bdea-db22546433ca"

os.environ['LUNARY_API_URL'] = LLMONITOR_API_URL
os.environ['LUNARY_PUBLIC_KEY'] = LLMONITOR_APP_ID
cwd = os.getcwd()


parent_dir = os.path.dirname(cwd)

print("Current Working Directory:", cwd)
print("Parent Directory:", parent_dir)
# HEX commands are too restrictive.
sys.path.append(os.path.abspath(parent_dir))

from nyx.data_generation.prompts import OPENAI_PREAMBLE


Current Working Directory: /Users/gtoth/PycharmProjects/demerzel/notebooks
Parent Directory: /Users/gtoth/PycharmProjects/demerzel


In [2]:
class CustomLLMonitorCallbackHandler(LunaryCallbackHandler):
    def on_llm_end(
        self,
        response: LLMResult,
        *,
        run_id: UUID,
        parent_run_id: Union[UUID, None] = None,
        **kwargs: Any,
    ) -> None:
        llm_output = response.llm_output or {}
        llm_output["token_usage"] = (response.llm_output or {}).get("usage", {})
        response.llm_output = llm_output
        super().on_llm_end(response, run_id=run_id, parent_run_id=parent_run_id, **kwargs)

ObservabilityCallback = CustomLLMonitorCallbackHandler(app_id=LLMONITOR_APP_ID, api_url=LLMONITOR_API_URL, verbose=True)

def retry(max_retries=3, retry_delay=1):
    """Decorator to retry a function or staticmethod if it raises an exception.

    :param max_retries: The maximum number of attempts to retry.
    :param retry_delay: The delay in seconds between retries.
    :return: A decorator that wraps the function or classmethod.
    """

    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            retries = 0
            while retries < max_retries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    retries += 1
                    print(f"Retrying {func.__name__} due to exception: {e}")
                    time.sleep(retry_delay)
            else:
                raise Exception(f"Maximum retries ({max_retries}) exceeded for {func.__name__}")

        return wrapper

    return decorator




@retry(max_retries=4, retry_delay=3)
def get_claude_evaluation(
    prompt_string, prompt_params, model_id="anthropic.claude-3-haiku-20240307-v1:0", model_kwargs=None
):
    model = ChatBedrock(
        region_name="us-east-1",
        model_id=model_id,
        model_kwargs=model_kwargs,
        callbacks=[ObservabilityCallback],
        tags=['Gregs-msc-evaluation'],
    )
    prompt = PromptTemplate.from_template(prompt_string)
    chain = prompt | model #| SimpleJsonOutputParser()

    return chain.invoke(prompt_params)

In [3]:
RATIONALES_SPLIT_STRING_EVAL = ("""Consider the coherence, accuracy, coverage, and overall quality of each summary and explain which one is better, in ONE or TWO SENTENCES. 
Then indicate the integer 1 or 2 for which summary is preferred.""")  # 77s on 50

TASK_WITH_COT_LEE_ET_AL_EVAL = (
    """\nText - {text}
Summary 1 - {summary1}
Summary 2 - {summary2}
"""
    + f"""
{RATIONALES_SPLIT_STRING_EVAL}
Rationale:"""
)



PROMPT_STR_TEMPLATE = (OPENAI_PREAMBLE + TASK_WITH_COT_LEE_ET_AL_EVAL).replace('<|im_start|>user', '').strip()

In [6]:
folder_labels = [#'hf-baseline', 
    'cot-baseline', 'insights-120-only-positive', 'insights-120-with-negatives', 'insights-240',
                 'insights-500']
phi_runs = ['8e9a5881d6c04fb6b20042829c9e5c3c', 'c745c16cb14147649de39c37c90db8f5', 'ccb3bd62cb454be49965529d56473fc5', 'dd6d7dba1dff4630b4cf7c32b4848adf', 'bbea9d66b20148a29fa41ff1cc586558', '64f781152e1c404aba665ad3707f4230'] 
qwen_runs = [#'e7b8abb8e9434263a979a386d8c9ee82', 
             'bd978ba45b24495db904ad394a8378a2', '95620e03ef074d429ff5df1070f5c715', '0c718a7ba4b84aca92c7fd3036f66e9f', 'd3ed4259ac15478b849dac198d37441e', '3a522f64e8914808b903647ac3c736fd']


In [ ]:
models = { 'qwen-2-7b': qwen_runs}  # 'phi-1-5': phi_runs,

# folder_labels = folder_labels[:1]
# phi_runs = phi_runs[:1]
# qwen_runs = qwen_runs[:1]
# del models['qwen-2-7b']


sub_path = 'results-section/03-reward-modelling-and-reinforcement-learning'
evaluation_results_dict = {}
costs_dict = {}
renaming_models = {'qwen-2-7b': 'qwen2-7b'}

for model, runs in models.items():
    print(model, runs)
    for run_id, run_type in zip(runs, folder_labels):

        print(model, run_type, run_id[:8])
        ending_path = (
            f'metrics/{renaming_models.get(model, model)}-ppo-telemetry-with-rm-scores.json'
        )
        with open(f'{parent_dir}/{sub_path}/{model}/{run_type}/{run_id}/{ending_path}') as f:
            ppo_results = json.load(f)

        evaluation_list = []
        costs_list = []
        for i in range(len(ppo_results['human-baseline-answers'])):
            if i % 50 == 0 and i >= 1:
                time.sleep(2)
            human_response = ppo_results['human-baseline-answers'][i]
            sft_response = ppo_results['sft-model-answers'][i]
            ppo_response = ppo_results['ppo-model-answers'][i]

            split_string = (
                'Summarize the following reddit post.'
                if model == 'phi-1-5'
                else 'Summarize the following reddit post:'
            )
            post = human_response.split('Summary:')[0].split(split_string)[1].strip()
            if post.endswith('assistant'):
                post = post[:-9]

            human_summary = human_response.split('Summary:')[1].strip()
            ppo_summary = ppo_response.split('Summary:')[1].strip()
            sft_summary = sft_response.split('Summary:')[1].strip()
            try:
                claude_response = get_claude_evaluation(
                    prompt_string=PROMPT_STR_TEMPLATE,
                    prompt_params={
                        'text': post,
                        'summary1': human_summary,
                        'summary2': ppo_summary,
                    },
                )
                claude_sft_response = get_claude_evaluation(
                    prompt_string=PROMPT_STR_TEMPLATE,
                    prompt_params={'text': post, 'summary1': sft_summary, 'summary2': ppo_summary},
                )
            except:
                print(f'\n\nProcessing stopped at run_id: {run_id} and index: {i}.')
                raise ValueError

            evaluation_list.append((claude_response.content, claude_sft_response.content))
            costs_list.append((claude_response, claude_sft_response))

        write_ending_path = f'metrics/human-vs-ppo-haiku-scores.json'
        write_path = f'{parent_dir}/{sub_path}/{model}/{run_type}/{run_id}/{write_ending_path}'
        with open(write_path, "w") as json_file:
            json.dump(
                {
                    'sft_evals': [i[1] for i in evaluation_list],
                    'human_evals': [i[0] for i in evaluation_list],
                    'human_prompt_cost': [
                        i[0].response_metadata['usage']['prompt_tokens'] * 0.00025 / 1_000
                        for i in costs_list
                    ],
                    'human_output_cost': [
                        i[0].response_metadata['usage']['completion_tokens'] * 0.00125 / 1_000
                        for i in costs_list
                    ],
                    'sft_prompt_cost': [
                        i[1].response_metadata['usage']['prompt_tokens'] * 0.00025 / 1_000
                        for i in costs_list
                    ],
                    'sft_output_cost': [
                        i[1].response_metadata['usage']['completion_tokens'] * 0.00125 / 1_000
                        for i in costs_list
                    ],
                },
                json_file,
                indent=4,
            )
        costs_dict[run_id] = costs_list
        evaluation_results_dict[run_id] = evaluation_list

        time.sleep(10)
        # print('finished a runId.')

print('done')


qwen-2-7b ['bd978ba45b24495db904ad394a8378a2', '95620e03ef074d429ff5df1070f5c715', '0c718a7ba4b84aca92c7fd3036f66e9f', 'd3ed4259ac15478b849dac198d37441e', '3a522f64e8914808b903647ac3c736fd']
qwen-2-7b cot-baseline bd978ba4
qwen-2-7b insights-120-only-positive 95620e03
Retrying get_claude_evaluation due to exception: Error raised by bedrock service: Could not connect to the endpoint URL: "https://bedrock-runtime.us-east-1.amazonaws.com/model/anthropic.claude-3-haiku-20240307-v1%3A0/invoke"
qwen-2-7b insights-120-with-negatives 0c718a7b
qwen-2-7b insights-240 d3ed4259


In [ ]:
claude_response.response_metadata['usage']['prompt_tokens']
claude_response.response_metadata['usage']['completion_tokens']